# Patient-Calibrated 3-Stage XGBoost (Fixed)

**Overview: The Patient-Calibrated Paradigm**

This notebook evaluates the efficacy of a "patient-calibrated" XGBoost machine learning pipeline for classifying a simplified 3-stage macro-architecture (Wake, NREM, REM) using exclusively wrist acceleration data.

**Methodological Note (v2 Fixes)**

All reported performance metrics in this notebook are derived from **out-of-fold cross-validation predictions** via `cross_val_predict` with `GroupKFold`. The final model fit is used only for feature importance extraction, never for evaluation. Post-prediction smoothing respects subject boundaries and uses rolling mode (not median) for categorical data. Cohen's Kappa is included as the standard sleep staging agreement metric.

## 1. Unzip and Load Data

### Subtask: Raw Data Extraction and Ingestion

This module handles the extraction and ingestion of the raw dataset. We systematically load the high-frequency wrist actigraphy (accelerometer x, y, z) and the corresponding Polysomnography (PSG) labels into isolated Pandas DataFrames. Critically, we intentionally exclude all photoplethysmography (heart rate) data files. This enforces our core research constraint: evaluating the predictive power of computational motion analysis isolated from physiological metrics.

In [ ]:
import zipfile
import os
import pandas as pd

# --- 1.A Dataset Extraction ---
zip_path = '/content/heartratedata.zip'
extract_path = '/content/heartratedata/'

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

base_path = '/content/heartratedata/motion-and-heart-rate-from-a-wrist-worn-wearable-and-labeled-sleep-from-polysomnography-1.0.0'
motion_dir = os.path.join(base_path, 'motion')
labels_dir = os.path.join(base_path, 'labels')

motion_list = []
labels_list = []
subject_ids = []

# --- 1.B Dynamic File Discovery ---
if os.path.exists(motion_dir):
    for filename in os.listdir(motion_dir):
        if filename.endswith('_acceleration.txt'):
            subject_id = filename.split('_')[0]
            subject_ids.append(subject_id)

# Fix 7: Deterministic subject ordering for reproducibility
subject_ids = sorted(subject_ids)

print(f"Discovered {len(subject_ids)} individual subjects.")

# --- 1.C Data Ingestion Loop ---
for subject_id in subject_ids:
    motion_file = os.path.join(motion_dir, f"{subject_id}_acceleration.txt")
    if os.path.exists(motion_file):
        try:
            df_m = pd.read_csv(motion_file, sep=' ', header=None, names=['timestamp', 'x', 'y', 'z'])
            df_m['subject_id'] = subject_id
            motion_list.append(df_m)
        except Exception as e:
            print(f"Error reading motion file for {subject_id}: {e}")

    label_file = os.path.join(labels_dir, f"{subject_id}_labeled_sleep.txt")
    if os.path.exists(label_file):
        try:
            df_l = pd.read_csv(label_file, sep=' ', header=None, names=['timestamp', 'label'])
            df_l['subject_id'] = subject_id
            labels_list.append(df_l)
        except Exception as e:
            print(f"Error reading label file for {subject_id}: {e}")

# --- 1.D DataFrame Concatenation ---
if motion_list:
    motion_df = pd.concat(motion_list, ignore_index=True)
    print("Motion Data successfully loaded into memory.")
else:
    raise ValueError("Critical Error: No Motion Data Found.")

if labels_list:
    labels_df = pd.concat(labels_list, ignore_index=True)
    print("PSG Labels successfully loaded into memory.")
else:
    raise ValueError("Critical Error: No Labels Data Found.")

## 2. Data Synchronization and Label Processing

### Subtask: Temporal Alignment and Macro-Class Grouping

High-frequency raw motion data cannot be directly trained against 30-second sleep epoch labels. This section standardizes the dataset by casting timestamps to compatible formats and employing a backward-filling merge_asof function with a 30-second tolerance guard. For this baseline model, we structurally group the clinical N1, N2, and N3 stages into a single "NREM" macro-class to address clinical practicality and mitigate the extreme class imbalances found in higher-resolution staging.

In [ ]:
# --- 2.A Sleep Stage Label Mapping ---
label_map = {
    0: 'Wake',
    1: 'NREM',
    2: 'NREM',
    3: 'NREM',
    5: 'REM'
}

labels_df = labels_df[labels_df['label'].isin(label_map.keys())].copy()
labels_df['sleep_stage'] = labels_df['label'].map(label_map)

# --- 2.B Data Type Standardization ---
labels_df['timestamp'] = labels_df['timestamp'].astype(float)
motion_df['subject_id'] = motion_df['subject_id'].astype(str)
labels_df['subject_id'] = labels_df['subject_id'].astype(str)

# --- 2.C Global Temporal Sorting ---
motion_df = motion_df.sort_values(by=['timestamp'])
labels_df = labels_df.sort_values(by=['timestamp'])

# --- 2.D Temporal Synchronization ---
# Fix 6: Added tolerance=30 to prevent stale label assignment
merged_df = pd.merge_asof(motion_df, labels_df, on='timestamp', by='subject_id',
                           direction='backward', tolerance=30)

merged_df = merged_df.dropna(subset=['sleep_stage'])

print("Class Distribution in Synchronized Dataset:")
print(merged_df['sleep_stage'].value_counts())

## 3. Time-Series Feature Engineering

### Subtask: Contextualizing Motion Data

Accelerometers suffer from the "Stillness Paradox" where REM and NREM stages can present identical instantaneous physical profiles (paralysis vs. deep rest). To address this, we engineer macro-temporal context. By calculating the Vector Magnitude (VM) and generating rolling temporal windows (2 and 5 minutes), we allow the model to recognize "sedentary streaks." We deliberately prune high-frequency noise metrics (Zero-Crossing Rates) to optimize computational efficiency.

In [ ]:
import numpy as np

# --- 3.A Vector Magnitude and Epoch Aggregation ---
merged_df['epoch_start'] = (merged_df['timestamp'] // 30) * 30
merged_df['vm'] = np.sqrt(merged_df['x']**2 + merged_df['y']**2 + merged_df['z']**2)

epoch_df = merged_df.groupby(['subject_id', 'epoch_start']).agg(
    mean_vm=('vm', 'mean'),
    std_vm=('vm', 'std'),
    sleep_stage=('sleep_stage', 'first')
).reset_index()

# --- 3.B Macro-Temporal Context (Rolling Windows) ---
epoch_df = epoch_df.sort_values(by=['subject_id', 'epoch_start'])

epoch_df['roll_mean_vm_2m'] = epoch_df.groupby('subject_id')['mean_vm'].transform(lambda x: x.rolling(window=4, min_periods=1).mean())
epoch_df['roll_std_vm_2m']  = epoch_df.groupby('subject_id')['std_vm'].transform(lambda x: x.rolling(window=4, min_periods=1).mean())

epoch_df['roll_mean_vm_5m'] = epoch_df.groupby('subject_id')['mean_vm'].transform(lambda x: x.rolling(window=10, min_periods=1).mean())
epoch_df['roll_std_vm_5m']  = epoch_df.groupby('subject_id')['std_vm'].transform(lambda x: x.rolling(window=10, min_periods=1).mean())

# --- 3.C Dynamic Rest-Activity Thresholding ---
def get_movement_mask(group):
    baseline = group['mean_vm'].quantile(0.05)
    return group['mean_vm'] > (baseline + 0.05)

# Fix 9: Added include_groups=False to prevent deprecation warning
epoch_df['is_movement'] = epoch_df.groupby('subject_id').apply(
    get_movement_mask, include_groups=False
).reset_index(level=0, drop=True)

epoch_df['last_movement_time'] = epoch_df['epoch_start'].where(epoch_df['is_movement']).groupby(epoch_df['subject_id']).ffill()
epoch_df['time_since_last_movement'] = epoch_df['epoch_start'] - epoch_df['last_movement_time'].fillna(epoch_df['epoch_start'])

# Memory cleanup
epoch_df.drop(columns=['is_movement', 'last_movement_time'], inplace=True)

# Fix 8: Targeted fillna instead of blanket fillna(0)
epoch_df['std_vm'] = epoch_df['std_vm'].fillna(0)
epoch_df['time_since_last_movement'] = epoch_df['time_since_last_movement'].ffill().fillna(0)
for col in [c for c in epoch_df.columns if c.startswith('roll_')]:
    epoch_df[col] = epoch_df[col].fillna(0)

## 4. XGBoost Pipeline and Cross-Validation

### Subtask: Gradient Boosting Architecture and Out-of-Fold Evaluation

We deploy an XGBoost classifier within an imbalanced-learn Pipeline with SMOTE for class balancing. Because XGBoost requires numerical target variables, we use a LabelEncoder. We use `cross_val_predict` with `GroupKFold` to generate **out-of-fold (OOF) predictions** that serve as the basis for all downstream evaluation. The final `pipeline.fit` is performed solely for feature importance extraction.

In [ ]:
import xgboost as xgb
from sklearn.model_selection import GroupKFold, cross_val_predict, cross_validate
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder

# --- 4.A Feature and Target Definition ---
features = [
    'mean_vm', 'std_vm', 'time_since_last_movement',
    'roll_mean_vm_2m', 'roll_std_vm_2m', 'roll_mean_vm_5m', 'roll_std_vm_5m'
]
target = 'sleep_stage'

X = epoch_df[features]

le = LabelEncoder()
y_encoded = le.fit_transform(epoch_df[target])
groups = epoch_df['subject_id']

# --- 4.B Model Pipeline Initialization ---
smote = SMOTE(random_state=42)

xgb_model = xgb.XGBClassifier(
    n_estimators=150,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='multi:softprob',
    num_class=3,
    random_state=42,
    tree_method='hist'
)

pipeline = Pipeline([('smote', smote), ('xgb', xgb_model)])

# --- 4.C Patient-Isolated Cross-Validation ---
cv = GroupKFold(n_splits=5)

# Fix 5: Report mean +/- std
cv_results = cross_validate(pipeline, X, y_encoded, groups=groups, cv=cv, scoring='f1_macro')
print(f"XGBoost (3-Stage) F1 Macro: {cv_results['test_score'].mean():.4f} +/- {cv_results['test_score'].std():.4f}")

# --- 4.D Out-of-Fold Predictions (Fix 1) ---
oof_probs = cross_val_predict(pipeline, X, y_encoded, groups=groups, cv=cv, method='predict_proba')

# Map class names for downstream use
class_names = le.classes_
classes = list(class_names)
y = epoch_df['sleep_stage']

# --- 4.E Final Model Fit (for feature importances ONLY) ---
pipeline.fit(X, y_encoded)
clf = pipeline.named_steps['xgb']
print("\nFinal model fitted for feature importance extraction only.")
print("All evaluation metrics below use out-of-fold predictions.")

## 5. Helper Functions and REM Threshold Optimization

### Subtask: Mitigating the "Stillness Paradox" with Proper Evaluation

We define reusable helper functions for vectorized threshold application and per-subject rolling mode smoothing. The threshold search operates exclusively on out-of-fold probabilities, preventing any data leakage. Rolling mode (not median) is used because sleep stages are categorical, making arithmetic operations like median mathematically invalid.

In [ ]:
from sklearn.metrics import f1_score
from scipy.stats import mode
import numpy as np
import pandas as pd

# ========== HELPER FUNCTIONS ==========

def apply_thresholds(probs, classes, thresh_rem):
    """Vectorized threshold application for 3-stage classification."""
    rem_idx = classes.index('REM')
    rem_mask = probs[:, rem_idx] >= thresh_rem
    default_probs = probs.copy()
    default_probs[:, rem_idx] = -1
    default_preds_idx = np.argmax(default_probs, axis=1)
    y_pred = np.where(rem_mask, 'REM', np.array(classes)[default_preds_idx])
    return y_pred

def smooth_predictions(y_pred_strings, subject_ids_array, stage_to_int, int_to_stage, window=5):
    """Per-subject rolling mode smoothing for categorical predictions."""
    def rolling_mode(series, w=window):
        return series.rolling(window=w, center=True, min_periods=1).apply(
            lambda x: mode(x, keepdims=False).mode
        )
    y_pred_ints = pd.Series(y_pred_strings).map(stage_to_int).values
    temp_df = pd.DataFrame({'subject_id': subject_ids_array, 'pred': y_pred_ints})
    temp_df['smoothed'] = temp_df.groupby('subject_id')['pred'].transform(
        lambda x: rolling_mode(x)
    ).astype(int)
    return temp_df['smoothed'].map(int_to_stage).values

# ========== STAGE MAPPINGS ==========

stage_to_int = {'NREM': 0, 'REM': 1, 'Wake': 2}
int_to_stage = {0: 'NREM', 1: 'REM', 2: 'Wake'}
subject_ids_array = epoch_df['subject_id'].values

# ========== THRESHOLD OPTIMIZATION (on OOF predictions) ==========

thresholds = np.arange(0.30, 0.71, 0.01)
best_threshold = 0.50
best_f1_macro = 0.0

print("Threshold optimization using out-of-fold predictions...\n")

for thresh in thresholds:
    y_pred = apply_thresholds(oof_probs, classes, thresh)
    y_pred_smooth = smooth_predictions(y_pred, subject_ids_array, stage_to_int, int_to_stage)
    current_f1 = f1_score(y, y_pred_smooth, average='macro')
    if current_f1 > best_f1_macro:
        best_f1_macro = current_f1
        best_threshold = thresh

print(f"Optimal REM threshold: {best_threshold:.2f} (OOF F1 Macro: {best_f1_macro:.4f})")

## 6. Final Evaluation Metrics and Visualization

### Subtask: Out-of-Fold Diagnostic Performance

This section applies the optimized threshold to the out-of-fold probability predictions and generates standard clinical diagnostic metrics. All numbers reported here reflect genuine generalization performance, as no prediction was made on data the model trained on.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, cohen_kappa_score

# --- 6.A Apply Best Threshold to OOF Probabilities ---
y_pred_thresh = apply_thresholds(oof_probs, classes, best_threshold)
y_pred_final = smooth_predictions(y_pred_thresh, subject_ids_array, stage_to_int, int_to_stage)

# --- 6.B Statistical Performance Report ---
print(f"3-Stage XGBoost Report (REM Thresh: {best_threshold:.2f}):")
print(f"  [All metrics from out-of-fold predictions]\n")
print(classification_report(y, y_pred_final))

# --- 6.C Cohen's Kappa (Fix 4) ---
kappa = cohen_kappa_score(y, y_pred_final)
print(f"Cohen's Kappa: {kappa:.4f}")

# --- 6.D Confusion Matrix (Fix 12: axis labels) ---
cm = confusion_matrix(y, y_pred_final, labels=classes)
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

plt.figure(figsize=(8, 6))
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=classes, yticklabels=classes)
plt.title('3-Stage XGBoost Confusion Matrix (Out-of-Fold)')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.tight_layout()
plt.show()

# --- 6.E Feature Importance (from final fit, NOT evaluation) ---
importances = clf.feature_importances_
fi_df = pd.DataFrame({'Feature': features, 'Importance': importances}).sort_values('Importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', hue='Feature', data=fi_df, palette='viridis', legend=False)
plt.title('XGBoost Feature Importance')
plt.tight_layout()
plt.show()

# --- 6.F Summary ---
report_dict = classification_report(y, y_pred_final, output_dict=True)
macro_f1 = report_dict['macro avg']['f1-score']
accuracy = report_dict['accuracy']

print(f"\nOut-of-Fold Summary: Accuracy={accuracy:.2%}, Macro F1={macro_f1:.4f}, Kappa={kappa:.4f}")
print("These metrics reflect generalization performance via GroupKFold cross-validation.")

## 7. Per-Subject Performance Breakdown

To assess variability across individuals, we compute F1 Macro for each subject and report mean and standard deviation. This validates whether "patient calibration" helps uniformly or benefits some subjects more than others.

In [ ]:
from sklearn.metrics import f1_score

subject_f1s = []
for sid in sorted(epoch_df['subject_id'].unique()):
    mask = epoch_df['subject_id'] == sid
    if mask.sum() == 0:
        continue
    y_true_subj = y.values[mask] if hasattr(y, 'values') else y[mask]
    y_pred_subj = y_pred_final[mask]
    f1 = f1_score(y_true_subj, y_pred_subj, average='macro', zero_division=0)
    subject_f1s.append({'subject_id': sid, 'f1_macro': f1})

subject_df = pd.DataFrame(subject_f1s)
print(f"Per-Subject F1 Macro: {subject_df['f1_macro'].mean():.4f} +/- {subject_df['f1_macro'].std():.4f}")
print(f"  Min: {subject_df['f1_macro'].min():.4f}  Max: {subject_df['f1_macro'].max():.4f}\n")
print(subject_df.to_string(index=False))

## 8. Hypnogram Visualization

Compare predicted vs. true sleep staging for one example subject. Hypnograms are the standard visualization in sleep medicine and provide intuitive assessment of temporal prediction accuracy.

In [ ]:
import matplotlib.pyplot as plt

# Pick the first subject deterministically
example_subject = sorted(epoch_df['subject_id'].unique())[0]
mask = (epoch_df['subject_id'] == example_subject).values

epochs = epoch_df.loc[mask, 'epoch_start'].values
y_true_subj = y.values[mask] if hasattr(y, 'values') else y[mask]
y_pred_subj = y_pred_final[mask]

stage_order = {'Wake': 2, 'REM': 1, 'NREM': 0}
ytick_labels = ['NREM', 'REM', 'Wake']

true_numeric = [stage_order[s] for s in y_true_subj]
pred_numeric = [stage_order[s] for s in y_pred_subj]

hours = (epochs - epochs[0]) / 3600.0

fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
axes[0].step(hours, true_numeric, where='mid', color='blue', linewidth=0.8)
axes[0].set_yticks(range(len(ytick_labels)))
axes[0].set_yticklabels(ytick_labels)
axes[0].set_title(f'True Hypnogram - Subject {example_subject}')
axes[0].set_ylabel('Sleep Stage')

axes[1].step(hours, pred_numeric, where='mid', color='orange', linewidth=0.8)
axes[1].set_yticks(range(len(ytick_labels)))
axes[1].set_yticklabels(ytick_labels)
axes[1].set_title(f'Predicted Hypnogram (OOF) - Subject {example_subject}')
axes[1].set_xlabel('Time (hours)')
axes[1].set_ylabel('Sleep Stage')

plt.tight_layout()
plt.show()